<a href="https://colab.research.google.com/github/kk7188048/MLpoject/blob/main/DeepSeek_R1_Deep_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

!pip install ollama

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [ ]:
!nvidia-smi

!df -h /

Sun Mar 22 14:37:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,452 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,939 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

!pip install ollama

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!which ollama
!ollama --version

/usr/local/bin/ollama


In [ ]:
!ollama list

Error: could not connect to ollama server, run 'ollama serve' to start it


In [ ]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

try:
    response = requests.get("http://localhost:11434")
    print("Ollama server is running!")
    print(f"Status: {response.status_code}")
except:
    print("Server not responding — wait 5 more seconds and retry")

Ollama server is running!
Status: 200


In [ ]:
!ollama pull deepseek-r1:7b

In [ ]:
!ollama list

NAME              ID              SIZE      MODIFIED               
deepseek-r1:7b    755ced02ce7b    4.7 GB    Less than a second ago    


When to use 1.5B: If the 7B download is taking forever, or if Colab didn't give you a T4. The 1.5B model still produces <think> blocks and you can learn all the same concepts — just with simpler reasoning traces.

In [ ]:
import requests
import json

def ask_r1(prompt: str, model: str = "deepseek-r1:7b") -> str:
    """Send a prompt to R1 and return raw response (including think block)."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    response.raise_for_status()
    return response.json()["response"]

raw_output = ask_r1(
    "What are 3 key things a researcher should know about AI hallucinations?",
    model="deepseek-r1:7b"
)

print(raw_output)

The three key areas a researcher should consider regarding AI hallucinations are:

1. **Generation Mechanisms**: Understanding how AI hallucinations are created through machine learning techniques such as Generative Adversarial Networks (GANs) and other algorithms that generate content not based on real data.

2. **Context and Reliability Evaluation**: Assessing the context in which these hallucinations appear and evaluating their reliability to ensure they align with real-world logic and appropriate scenarios.

3. **Detection and Prevention of Harmfulness**: Implementing methods to identify and mitigate the use of AI hallucinations that may be misleading or harmful, such as detecting fake news or deceptive imagery.


In [ ]:
import requests
import json
import re

def ask_r1(prompt, model="deepseek-r1:7b", temperature: float = 0.6):
    """
    Uses /api/chat instead of /api/generate.

    WHY THIS MATTERS:
    - /api/generate = raw text completion. No chat template applied.
      R1 doesn't know to use <think> tags. Behaves like GPT-2.
    - /api/chat = applies the model's chat template (ChatML format).
      R1 sees [INST] markers and activates its thinking mode properly.
    """
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [
                {
                    "role": "system",
                    "content": "You are a helpful research assistant. Think carefully and thoroughly before responding."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "stream": True,
            "options": {
                "temperature": temperature,
                "num_predict": 2048,
            }
        },
        timeout=300,
        stream=True
    )

    full_response_content = ""
    full_response_thinking = ""

    for chunk in response.iter_lines():
        if chunk:
            decoded_chunk = chunk.decode('utf-8')
            try:
                json_chunk = json.loads(decoded_chunk)
                message_content = json_chunk.get("message", {}).get("content", "")
                message_thinking = json_chunk.get("message", {}).get("thinking", "")

                if message_content:
                    full_response_content += message_content
                if message_thinking:
                    full_response_thinking += message_thinking


            except json.JSONDecodeError:
                pass

    final_message = {
        "role": "assistant",
        "content": full_response_content,
        "thinking": full_response_thinking
    }

    print("FULL message object (from ask_r1 - streamed result):", json.dumps(final_message, indent=2))
    return final_message


def parse_r1_output(message_obj: dict) -> dict:
    """Robust parser — extracts content and thinking directly from the message object."""

    content = message_obj.get("content", "")
    thinking = message_obj.get("thinking", "")

    has_think_block = bool(thinking)

    return {
        "thinking": thinking.strip(),
        "answer": content.strip(),
        "think_tokens": len(thinking.split()) if thinking else 0,
        "answer_tokens": len(content.split()),
        "has_think_block": has_think_block
    }

In [ ]:
prompt = """A researcher is investigating whether AI language models can truly reason
or are just doing pattern matching. What are the strongest arguments on both sides,
and what experiments could settle the debate?"""

full_response_message = ask_r1(prompt)

result = parse_r1_output(full_response_message)
print("Parse r1 output",result)

if not result["has_think_block"]:
    print("Still no think block — run the debug cell below")
else:
    print(f"Think block: {result['think_tokens']} words")
    print(f"Answer:      {result['answer_tokens']} words\n")
    print("=" * 50)
    print("THINKING TRACE:")
    print("=" * 50)
    print(result["thinking"])
    print("\n" + "=" * 50)
    print("FINAL ANSWER:")
    print("=" * 50)
    print(result["answer"])

KeyboardInterrupt: 

In [ ]:
!ollama show deepseek-r1:7b

  Model
    architecture        qwen2     
    parameters          7.6B      
    context length      131072    
    embedding length    3584      
    quantization        Q4_K_M    

  Capabilities
    completion    
    thinking      

  Parameters
    stop    "<｜begin▁of▁sentence｜>"    
    stop    "<｜end▁of▁sentence｜>"      
    stop    "<｜User｜>"                 
    stop    "<｜Assistant｜>"            

  License
    MIT License                    
    Copyright (c) 2023 DeepSeek    
    ...                            



In [ ]:
!ollama --version

ollama version is 0.18.2


In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "deepseek-r1:7b",
        "messages": [{"role": "user", "content": "What is 1+1?"}],
        "stream": False,
        "options": {"temperature": 0.6, "num_predict": 2048}
    },
    timeout=300
)

data = response.json()

print("FULL message object:")
import json
print(json.dumps(data.get("message", {}), indent=2))

FULL message object:
{
  "role": "assistant",
  "content": "Sure! Let's solve the simple addition problem step by step.\n\n**Problem:**  \nWhat is \\(1 + 1\\)?\n\n**Solution:**  \nTo find the sum of 1 and 1:\n\n\\[\n1 + 1 = 2\n\\]\n\nSo, the answer is \\(\\boxed{2}\\).",
  "thinking": "To solve the problem of adding one and one, I'll begin by identifying the numbers involved.\n\nNext, I'll perform the addition operation: 1 plus 1 equals 2.\n"
}


In [ ]:
!pip install tavily-python -q

import os
os.environ["TAVILY_API_KEY"] = "tvly-YOUR-KEY-HERE"

from tavily import TavilyClient
client = TavilyClient("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")

result = client.search("DeepSeek R1 reasoning model 2024", max_results=2)
print(f"Search returned {len(result['results'])} results")
print(result['results'][0]['title'])
print(result['results'][0]['url'])

Search returned 2 results
Reasoning Model (deepseek-reasoner)
https://api-docs.deepseek.com/guides/reasoning_model


In [ ]:
from tavily import TavilyClient
import os

tavily = TavilyClient("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")

def web_search(query: str, max_results: int = 3) -> str:
    """
    Search the web and return clean text results.
    Returns a formatted string that R1 can read inside its context.
    """
    results = tavily.search(
        query=query,
        max_results=max_results,
        search_depth="basic"
    )

    if not results["results"]:
        return "No results found for this query."

    formatted = []
    for i, r in enumerate(results["results"], 1):
        formatted.append(
            f"[Source {i}] {r['title']}\n"
            f"URL: {r['url']}\n"
            f"Content: {r['content'][:400]}..."
        )

    return "\n\n".join(formatted)


print(web_search("what is chain of thought prompting in LLMs"))

[Source 1] What is Chain of Thought (CoT) Prompting? | NVIDIA Glossary
URL: https://www.nvidia.com/en-us/glossary/cot-prompting/
Content: Accelerated, containerized AI models and SDKs. High performance GeForce RTX PCs, purpose-built for creators. Accelerate AI and HPC workloads with NVIDIA GPU Cloud solutions. Accelerate AI and HPC workloads with NVIDIA GPU Cloud solutions. The automation in the Chain of Thought process is a breakthrough in AI's ability to handle complex reasoning tasks. ## Generative AI, Scaling Laws, and Chain of ...

[Source 2] Chain-of-Thought Prompting Elicits Reasoning in Large Language ...
URL: https://arxiv.org/abs/2201.11903
Content: # Title:Chain-of-Thought Prompting Elicits Reasoning in Large Language Models. View a PDF of the paper titled Chain-of-Thought Prompting Elicits Reasoning in Large Language Models, by Jason Wei and 8 other authors. > Abstract:We explore how generating a chain of thought -- a series of intermediate reasoning steps -- significantly 

In [ ]:
REACT_SYSTEM_PROMPT = """You are a research assistant. Answer the user's question by searching the web.

Follow this exact format for EVERY response:

Thought: [reason about what to search for next, or whether you have enough info]
Action: search["your search query here"]

OR when you have enough information:

Thought: [reason that you have enough information to answer]
Action: finish["your complete final answer here"]

Rules:
- Always start with a Thought
- Always follow a Thought with an Action
- Use search[] when you need more information
- Use finish[] only when you can give a complete, well-sourced answer
- You may search up to 5 times before you must finish
- Cite sources in your final answer"""

print("System prompt loaded. Length:", len(REACT_SYSTEM_PROMPT), "chars")

System prompt loaded. Length: 682 chars


In [ ]:
import re

def react_agent(question: str, max_steps: int = 6) -> dict:
    """
    ReAct loop: R1 thinks and acts in turns.
    Each turn: R1 outputs Thought + Action
    If action is search -> we run the search and feed results back
    If action is finish -> we return the answer
    """
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    steps_log = []
    search_count = 0

    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")

        prompt = ""
        for m in messages:
            if m["role"] == "system":
                prompt += f"SYSTEM: {m['content']}\n\n"
            elif m["role"] == "user":
                prompt += f"USER: {m['content']}\n\n"
            elif m["role"] == "assistant":
                prompt += f"ASSISTANT: {m['content']}\n\n"

        prompt += "ASSISTANT:"

        raw = ask_r1(prompt)
        result = parse_r1_output(raw)
        agent_text = result["answer"]

        print(f"R1 says:\n{agent_text[:300]}")
        steps_log.append({"step": step+1, "agent": agent_text})

        messages.append({"role": "assistant", "content": agent_text})


        finish_match = re.search(r'Action:\s*finish["]?([^"]+)?["]?\]', agent_text, re.IGNORECASE | re.DOTALL)
        if finish_match:
            final_answer = finish_match.group(1).strip()
            print("\nR1 decided to finish!")
            return {
                "question": question,
                "answer": final_answer,
                "steps": steps_log,
                "searches": search_count
            }

        search_match = re.search(r'Action:\s*search["]?([^"]+)?["]?\]', agent_text, re.IGNORECASE)
        if search_match:
            query = search_match.group(1).strip()
            search_count += 1
            print(f"R1 searching for: '{query}'")

            search_results = web_search(query)
            observation = f"Observation: Here are the search results for '{query}':\n\n{search_results}"
            messages.append({"role": "user", "content": observation})
            print(f"Search returned {len(search_results)} chars of results")
            continue
        messages.append({
            "role": "user",
            "content": "Please continue. Remember to output: Thought: [reasoning] then Action: search[\"query\"] or Action: finish[\"answer\"]"
        })

    print("\nMax steps reached — asking for final answer")
    messages.append({"role": "user", "content": 'You have reached the search limit. Summarize what you found. Use Action: finish[\"your answer\"]'})
    raw = ask_r1("\n".join([m["content"] for m in messages[-3:]]))
    result = parse_r1_output(raw)
    return {
        "question": question,
        "answer": result["answer"],
        "steps": steps_log,
        "searches": search_count
    }

In [ ]:
result = react_agent(
    "What are the latest breakthroughs in AI reasoning models in 2024-2025?"
)

print("\n" + "="*60)
print("FINAL ANSWER:")
print("="*60)
print(result["answer"])
print(f"\n[Used {result['searches']} web searches in {len(result['steps'])} steps]")


--- Step 1 ---
FULL message object (from ask_r1 - streamed result): {
  "role": "assistant",
  "content": "The latest breakthroughs in AI reasoning models for 2024-2025 are expected to focus on several key areas:\n\n1. **Large Language Models (LLMs) Upgrades**: The release of GPT-5 by OpenAI is anticipated, building on the success of previous models like GPT-4. This update may include enhanced reasoning capabilities and improved efficiency.\n\n2. **Enhanced Reasoning Capabilities**: Developments in neural architectures from institutions like DeepMind could lead to more logical reasoning, moving beyond surface-level text analysis.\n\n3. **Transfer Learning Innovations**: Advances such as those from Meta and Google aim to improve the transfer of knowledge across diverse domains, making AI more adaptable and versatile.\n\n4. **Explainable AI (XAI) Progress**: Research groups like Microsoft and IBM are likely introducing new methods to make AI reasoning transparent, enhancing trust and ac

In [ ]:
import json
import re

PLANNER_PROMPT = """You are a research planning expert.

Your job: decompose a broad research question into focused sub-questions.

Rules:
- Generate exactly 3 to 5 sub-questions
- Each sub-question must be specific and independently searchable
- Sub-questions must NOT overlap with each other
- Cover different angles: background, current state, challenges, future
- Respond ONLY with a valid JSON array of strings — no explanation, no markdown

Example output:
["What is the history of X?", "How does X work technically?", "What are the main challenges with X?"]"""


def decompose_query(main_question: str) -> list[str]:
    """
    Ask R1 to break a big research question into focused sub-questions.
    Returns a Python list of sub-question strings.
    """
    prompt = f"""SYSTEM: {PLANNER_PROMPT}

USER: Main research question: {main_question}

ASSISTANT:"""

    raw = ask_r1(prompt)
    result = parse_r1_output(raw)
    answer = result["answer"].strip()

    answer = re.sub(r'^```(?:json)?\s*', '', answer)
    answer = re.sub(r'\s*```$', '', answer)

    try:
        sub_questions = json.loads(answer)
        print(f"Decomposed into {len(sub_questions)} sub-questions:")
        for i, q in enumerate(sub_questions, 1):
            print(f"  {i}. {q}")
        return sub_questions
    except json.JSONDecodeError:
        print("JSON parse failed — extracting manually")

        matches = re.findall(r'"([^"]+)"', answer)
        return matches[:5] if matches else [main_question]


subs = decompose_query(
    "What are the key challenges and breakthroughs in AI reasoning models?"
)

FULL message object (from ask_r1 - streamed result): {
  "role": "assistant",
  "content": "[\"What are the primary components of AI reasoning models?\", \"How do AI reasoning models function technically and computationally?\", \"What are the key challenges faced by AI reasoning models in real-world applications?\", \"What are the recent breakthroughs in improving AI reasoning capabilities?\", \"Where is the future of AI reasoning technology heading?\"]",
  "thinking": "Okay, I need to break down the user's main research question into focused sub-questions. The main question is about the key challenges and breakthroughs in AI reasoning models.\n\nFirst, I'll think about what \"AI reasoning models\" encompass. They might involve how these models are structured, their capabilities, limitations, etc. So, starting with an overview makes sense\u2014what exactly are they?\n\nNext, understanding how they function technically would be a good sub-question. It's specific and search-related.\n\nC

In [ ]:
def research_all_subquestions(sub_questions: list[str]) -> dict:
    """
    Run react_agent() on every sub-question.
    Sequential (not async) — simpler, reliable on Colab free tier.
    Returns a dict: {sub_question -> findings_text}
    """
    all_findings = {}

    for i, question in enumerate(sub_questions, 1):
        print(f"\n{'='*55}")
        print(f"Researching sub-question {i}/{len(sub_questions)}:")
        print(f"  '{question}'")
        print('='*55)

        result = react_agent(question, max_steps=4)

        all_findings[question] = {
            "answer": result["answer"],
            "searches": result["searches"],
            "steps": len(result["steps"])
        }

        print(f"Done. ({result['searches']} searches, answer: {len(result['answer'])} chars)")

    return all_findings

print("research_all_subquestions() ready")
print("Will run react_agent() once per sub-question sequentially.")

research_all_subquestions() ready
Will run react_agent() once per sub-question sequentially.


In [ ]:
SYNTHESIZER_PROMPT = """You are an expert research writer and analyst.

You have been given a main research question and findings from multiple focused sub-investigations.

Your job: write a comprehensive, well-structured research report.

The report must include:
1. Executive Summary (2-3 sentences capturing the key insight)
2. Key Findings (one section per sub-question investigated)
3. Synthesis (how the findings connect — patterns, contradictions, surprises)
4. Research Gaps (what remains unknown or needs further investigation)
5. Conclusion

Requirements:
- Be specific — use facts and details from the findings, not vague statements
- Flag any contradictions between sources
- Note confidence level for claims (high / medium / low)
- Length: 400-600 words total
- Do NOT make up information — only use what is in the provided findings"""

print("Synthesizer prompt loaded:", len(SYNTHESIZER_PROMPT), "chars")

Synthesizer prompt loaded: 821 chars


In [ ]:
def synthesize_report(main_question: str, all_findings: dict) -> str:
    """
    Take all sub-question findings and produce a structured research report.
    This is a SINGLE R1 call — but with a rich, dense context.
    """
    findings_text = ""
    for i, (question, data) in enumerate(all_findings.items(), 1):
        findings_text += f"""
--- Finding {i} ---
Sub-question: {question}
Answer: {data['answer']}
(Based on {data['searches']} web searches)
"""

    synthesis_prompt = f"""SYSTEM: {SYNTHESIZER_PROMPT}

USER:
MAIN RESEARCH QUESTION:
{main_question}

FINDINGS FROM SUB-INVESTIGATIONS:
{findings_text}

Please write the comprehensive research report now.

ASSISTANT:"""

    print("Sending findings to R1 for synthesis...")
    print(f"Context size: ~{len(synthesis_prompt)} chars")

    raw = ask_r1(synthesis_prompt)
    result = parse_r1_output(raw)

    print(f"Synthesis complete. Report: {len(result['answer'].split())} words")
    print(f"R1 thinking during synthesis: {result['think_tokens']} words")

    return result["answer"]


print("synthesize_report() ready")

synthesize_report() ready


In [ ]:
def deep_research(question: str) -> dict:
    """
    The complete pipeline:
      Step A: Decompose question into sub-questions
      Step B: Research each sub-question with web search
      Step C: Synthesize all findings into a report
    """
    print("\n" + "#"*60)
    print("DEEP RESEARCH PIPELINE STARTING")
    print("#"*60)
    print(f"Question: {question}\n")

    print("[ STEP A ] Decomposing question into sub-questions...")
    sub_questions = decompose_query(question)

    print(f"\n[ STEP B ] Researching {len(sub_questions)} sub-questions...")
    all_findings = research_all_subquestions(sub_questions)

    print("\n[ STEP C ] Synthesizing findings into report...")
    report = synthesize_report(question, all_findings)

    total_searches = sum(f["searches"] for f in all_findings.values())
    print(f"\nPipeline complete!")
    print(f"  Sub-questions researched: {len(sub_questions)}")
    print(f"  Total web searches: {total_searches}")
    print(f"  Report length: {len(report.split())} words")

    return {
        "question": question,
        "sub_questions": sub_questions,
        "findings": all_findings,
        "report": report,
        "total_searches": total_searches
    }


print("deep_research() pipeline ready.")
print("Run the next cell to execute it on a real question.")

deep_research() pipeline ready.
Run the next cell to execute it on a real question.


In [ ]:
research_result = deep_research(
    "What are the key challenges and breakthroughs in AI reasoning models, "
    "and how should developers approach building applications with them?"
)

print("\n" + "="*60)
print("RESEARCH REPORT")
print("="*60)
print(research_result["report"])
print("\n" + "="*60)
print(f"Total searches used: {research_result['total_searches']}")
print(f"Sub-questions: {len(research_result['sub_questions'])}")


############################################################
DEEP RESEARCH PIPELINE STARTING
############################################################
Question: What are the key challenges and breakthroughs in AI reasoning models, and how should developers approach building applications with them?

[ STEP A ] Decomposing question into sub-questions...
FULL message object (from ask_r1 - streamed result): {
  "role": "assistant",
  "content": "[\"What is the history and evolution of AI reasoning models?\", \"How do AI reasoning models work technically?\", \"What are existing solutions and innovations in AI reasoning models?\", \"Why are certain challenges difficult to address in AI reasoning models?\", \"What best practices should developers follow when building applications with AI reasoning models?\"]",
  "thinking": "Okay, so I need to break down this big research question into smaller sub-questions. The main question is about AI reasoning models\u2014specifically their key challe

In [ ]:
print("ask_r1() function has been updated in cell n2eBlsjhnBUD to support temperature.")
print("This cell (Cu0OybmO67Uo) is now redundant and serves as a placeholder.")

ask_r1() updated with temperature support.
Default temperature: 0.6 (R1 recommended)
Self-consistency runs will use: 0.7
Deterministic (JSON) runs will use: 0.0


In [ ]:
def build_findings_text(all_findings: dict) -> str:
    """
    Rebuild the findings string from the stored research result.
    Extracted as its own function so both synthesizer and
    self-consistency runner can use it without code duplication.
    """
    text = ""
    for i, (question, data) in enumerate(all_findings.items(), 1):
        text += f"""
--- Finding {i} ---
Sub-question: {question}
Answer: {data['answer']}
(Based on {data['searches']} web searches)
"""
    return text


def run_self_consistency(
    main_question: str,
    all_findings: dict,
    n: int = 3
) -> tuple[list, list]:
    """
    Run the synthesizer N times with temperature=0.7.
    Returns (list_of_reports, list_of_think_blocks).

    Why temperature=0.7?
    At 0.0: R1 always picks the highest-probability token → identical output every run.
    At 0.7: R1 samples from a distribution → meaningfully different phrasings/structures.
    This lets us measure whether the *substance* is consistent even if wording varies.
    """
    findings_text = build_findings_text(all_findings)

    synthesis_prompt = f"""SYSTEM: {SYNTHESIZER_PROMPT}

USER:
MAIN RESEARCH QUESTION:
{main_question}

FINDINGS FROM SUB-INVESTIGATIONS:
{findings_text}

Please write the comprehensive research report now.

A:"""

    reports = []
    think_blocks = []

    for i in range(n):
        print(f"  Run {i+1}/{n}...", end=" ", flush=True)
        raw = ask_r1(synthesis_prompt, temperature=0.7)
        result = parse_r1_output(raw)
        reports.append(result["answer"])
        think_blocks.append(result["thinking"])
        print(f"{len(result['answer'].split())} words | think: {result['think_tokens']} words")

    return reports, think_blocks


print("run_self_consistency() ready — will call synthesizer 3x with temperature=0.7")

run_self_consistency() ready — will call synthesizer 3x with temperature=0.7


In [ ]:
def jaccard_similarity(text_a: str, text_b: str) -> float:
    """
    Measures word-level overlap between two texts.
    Returns a float between 0 (no overlap) and 1 (identical).

    Jaccard = |intersection| / |union|

    Why Jaccard and not cosine similarity?
    Cosine needs numpy + word vectors. Jaccard is pure Python,
    zero dependencies, and works well for comparing same-topic reports
    where vocabulary overlap is the key signal.
    """
    words_a = set(text_a.lower().split())
    words_b = set(text_b.lower().split())

    stopwords = {
        'the','a','an','is','are','was','were','be','been','being',
        'have','has','had','do','does','did','will','would','should',
        'may','might','can','could','that','this','these','those',
        'and','but','or','nor','for','yet','so','in','on','at','to',
        'of','by','as','it','its','with','from','into','about','than'
    }
    words_a -= stopwords
    words_b -= stopwords

    if not words_a or not words_b:
        return 0.0

    intersection = len(words_a & words_b)
    union = len(words_a | words_b)
    return round(intersection / union, 4)


def pick_best_report(reports: list) -> tuple[int, list, float]:
    """
    Selects the most representative report from N runs.

    Strategy: pick the report with highest average pairwise similarity
    to all other reports. This is the "centroid" of the distribution —
    the most typical, least-outlier output.

    Returns: (best_index, avg_similarities_list, consistency_score)
    """
    n = len(reports)
    avg_sims = []

    print("\nPairwise similarity matrix:")
    for i in range(n):
        row_sims = []
        for j in range(n):
            if i != j:
                s = jaccard_similarity(reports[i], reports[j])
                row_sims.append(s)
        avg = round(sum(row_sims) / len(row_sims), 4)
        avg_sims.append(avg)
        print(f"  Report {i+1}: avg similarity to others = {avg:.4f}")

    all_pairs = []
    for i in range(n):
        for j in range(i+1, n):
            all_pairs.append(jaccard_similarity(reports[i], reports[j]))
    consistency = round(sum(all_pairs) / len(all_pairs), 4)

    best_idx = avg_sims.index(max(avg_sims))
    print(f"\nBest report: #{best_idx+1} (most representative)")
    print(f"Overall consistency score: {consistency:.4f}")
    print(f"  (0.0 = completely different, 1.0 = identical)")
    print(f"  Typical good range: 0.25 – 0.55 for 400-word reports")

    return best_idx, avg_sims, consistency


print("jaccard_similarity() and pick_best_report() ready")

jaccard_similarity() and pick_best_report() ready


In [ ]:
def score_coverage(report: str, sub_questions: list) -> dict:
    """
    For each sub-question, extract its key content words and check
    whether they appear in the report.

    Why key words and not the full question?
    The report won't copy sub-questions verbatim — it paraphrases.
    Key content words (non-stopwords, len > 3) survive paraphrasing.

    Threshold: if >= 40% of a sub-question's key terms appear in the
    report, that sub-question is "covered".
    """
    report_lower = report.lower()

    STOPWORDS = {
        'what','are','the','how','does','when','where','which','have',
        'been','that','with','this','from','they','will','your','more',
        'about','than','into','its','for','and','but','not','key','main',
        'latest','recent','current','approach','building','applications',
        'using','most','important','between','during','after','before'
    }

    per_question = []
    covered_count = 0

    for q in sub_questions:

        key_terms = [
            w.strip('?.,').lower()
            for w in q.split()
            if len(w.strip('?.,')) > 3 and w.strip('?.,').lower() not in STOPWORDS
        ]

        if not key_terms:
            per_question.append({"question": q, "covered": True, "match_rate": 1.0})
            covered_count += 1
            continue

        found = [t for t in key_terms if t in report_lower]
        match_rate = round(len(found) / len(key_terms), 3) if key_terms else 0
        covered = match_rate >= 0.40

        if covered:
            covered_count += 1

        per_question.append({
            "question": q[:60] + "..." if len(q) > 60 else q,
            "key_terms": key_terms,
            "found_terms": found,
            "match_rate": match_rate,
            "covered": covered
        })

    overall = round(covered_count / len(sub_questions), 3) if sub_questions else 0

    return {
        "score": overall,
        "covered": covered_count,
        "total": len(sub_questions),
        "per_question": per_question
    }


print("score_coverage() ready")
print("Threshold: sub-question is 'covered' if >= 40% of its key terms appear in report")

score_coverage() ready
Threshold: sub-question is 'covered' if >= 40% of its key terms appear in report


In [ ]:
def score_faithfulness(report: str, all_findings: dict) -> dict:
    """
    Splits report into individual sentences, then checks whether
    each sentence has sufficient word overlap with the findings text.

    Why word overlap and not a neural NLI model?
    NLI (Natural Language Inference) models are accurate but require
    ~500MB downloads and GPU. Word overlap is a reasonable proxy:
    if a sentence in the report contains words that appeared in the
    findings, it's likely grounded in those findings.

    Supported threshold: >= 30% of a sentence's content words must
    appear in the combined findings text.
    """
    all_findings_text = " ".join(
        data["answer"] for data in all_findings.values()
    ).lower()

    STOPWORDS = {
        'the','a','an','is','are','was','were','be','been','this',
        'that','these','those','it','its','they','their','there',
        'and','but','or','in','on','at','to','of','by','for','with',
        'as','from','into','have','has','had','will','would','can',
        'also','which','who','what','how','when','where','while'
    }

    raw_sentences = re.split(r'(?<=[.!?])\s+|\\n', report)
    sentences = [s.strip() for s in raw_sentences if len(s.strip()) > 25]

    results = []
    supported_count = 0

    for sentence in sentences:

        words = set(
            w.strip('.,!?;:"()[]').lower()
            for w in sentence.split()
            if len(w.strip('.,!?;:"()[]')) > 3
        ) - STOPWORDS

        if not words:
            continue

        found_in_findings = sum(1 for w in words if w in all_findings_text)
        overlap_ratio = round(found_in_findings / len(words), 3) if words else 0
        supported = overlap_ratio >= 0.30

        if supported:
            supported_count += 1

        results.append({
            "sentence": sentence[:80] + "..." if len(sentence) > 80 else sentence,
            "overlap": overlap_ratio,
            "supported": supported
        })

    total = len(results)
    score = round(supported_count / total, 3) if total else 0

    return {
        "score": score,
        "supported_sentences": supported_count,
        "total_sentences": total,
        "unsupported": [r for r in results if not r["supported"]],
        "per_sentence": results
    }


print("score_faithfulness() ready")
print("Threshold: sentence is 'supported' if >= 30% content words appear in findings")

score_faithfulness() ready
Threshold: sentence is 'supported' if >= 30% content words appear in findings


In [ ]:
def score_reasoning_depth(think_block: str) -> dict:
    """
    Analyses the synthesis think block across 3 dimensions:

    1. Length — longer thinking generally = more thorough reasoning
       (but with diminishing returns; normalised at 600 words)

    2. Self-correction — phrases like "wait", "actually", "let me
       reconsider" indicate R1 caught its own errors. This is the
       most valuable reasoning signal — it means R1 is actively
       auditing its own output rather than just generating.

    3. Cross-referencing — phrases indicating R1 connected ideas
       across different findings rather than summarising each
       finding independently. This is what "synthesis" means.

    Returns a composite score 0.0–1.0 with component breakdown.
    """
    if not think_block or len(think_block.strip()) < 10:
        return {"score": 0.0, "length": 0, "corrections": 0, "cross_refs": 0,
                "length_score": 0.0, "correction_score": 0.0, "crossref_score": 0.0}

    think_lower = think_block.lower()
    word_count = len(think_block.split())

    length_score = round(min(word_count / 600, 1.0), 3)

    correction_phrases = [
        'wait,', 'wait —', 'actually,', 'actually —', 'hmm,',
        'let me reconsider', 'on second thought', 'i need to correct',
        'i made an error', 'let me re-read', 'more carefully',
        "i should reconsider", "that's not right", "i was wrong",
        'let me think again', 'hold on', 'no, actually', 'but wait'
    ]
    correction_hits = [p for p in correction_phrases if p in think_lower]
    correction_score = round(min(len(correction_hits) / 3, 1.0), 3)

    crossref_phrases = [
        'finding', 'sub-question', 'earlier i noted', 'as mentioned',
        'connects to', 'relates to', 'in contrast', 'compared to',
        'both sources', 'contradicts', 'consistent with', 'aligns with',
        'this ties', 'taken together', 'across all', 'collectively',
        'one finding', 'another finding', 'this is consistent',
        'this contradicts', 'looking at all'
    ]
    crossref_hits = [p for p in crossref_phrases if p in think_lower]
    crossref_score = round(min(len(crossref_hits) / 4, 1.0), 3)

    composite = round(
        length_score     * 0.40 +
        correction_score * 0.35 +
        crossref_score   * 0.25,
        3
    )

    return {
        "score": composite,
        "word_count": word_count,
        "corrections_found": correction_hits,
        "crossrefs_found": crossref_hits,
        "length_score": length_score,
        "correction_score": correction_score,
        "crossref_score": crossref_score
    }


print("score_reasoning_depth() ready")
print("Weights: length=40%, self-correction=35%, cross-referencing=25%")

score_reasoning_depth() ready
Weights: length=40%, self-correction=35%, cross-referencing=25%


In [ ]:
def evaluate_research_output(research_result: dict, n_consistency_runs: int = 3) -> dict:
    """
    Full evaluation pipeline for a deep_research() output dict.

    Takes the dict returned by deep_research() from Step 3.
    Runs self-consistency (N=3 additional synthesizer calls) and
    all 4 evaluation metrics. Returns a structured scores dict.
    """
    question     = research_result["question"]
    sub_questions = research_result["sub_questions"]
    all_findings = research_result["findings"]
    report       = research_result["report"]

    print("\n" + "="*60)
    print("EVALUATION STARTING")
    print("="*60)

    print("\n[1/4] Running self-consistency (3 synthesis runs)...")
    reports, think_blocks = run_self_consistency(
        question, all_findings, n=n_consistency_runs
    )
    best_idx, avg_sims, consistency_score = pick_best_report(reports)
    best_report = reports[best_idx]
    best_think  = think_blocks[best_idx]

    print("\n[2/4] Scoring coverage...")
    coverage = score_coverage(best_report, sub_questions)
    print(f"  Coverage: {coverage['covered']}/{coverage['total']} sub-questions")
    print(f"  Score: {coverage['score']:.1%}")

    print("\n[3/4] Scoring faithfulness...")
    faithfulness = score_faithfulness(best_report, all_findings)
    print(f"  Supported: {faithfulness['supported_sentences']}/{faithfulness['total_sentences']} sentences")
    print(f"  Score: {faithfulness['score']:.1%}")

    print("\n[4/4] Scoring reasoning depth...")
    depth = score_reasoning_depth(best_think)
    print(f"  Think block: {depth['word_count']} words")
    print(f"  Self-corrections found: {len(depth['corrections_found'])}")
    print(f"  Cross-references found: {len(depth['crossrefs_found'])}")
    print(f"  Score: {depth['score']:.3f} / 1.000")

    overall = round(
        faithfulness["score"]  * 0.35 +
        coverage["score"]      * 0.30 +
        consistency_score      * 0.20 +
        depth["score"]         * 0.15,
        3
    )

    print("\n" + "="*60)
    print(f"OVERALL QUALITY SCORE: {overall:.1%}")
    print("="*60)

    return {
        "best_report":       best_report,
        "all_reports":       reports,
        "consistency_score": consistency_score,
        "coverage":          coverage,
        "faithfulness":      faithfulness,
        "reasoning_depth":   depth,
        "overall_score":     overall
    }


print("evaluate_research_output() ready")
print("Metric weights: faithfulness=35%, coverage=30%, consistency=20%, depth=15%")

evaluate_research_output() ready
Metric weights: faithfulness=35%, coverage=30%, consistency=20%, depth=15%


In [ ]:
def print_eval_dashboard(eval_result: dict):
    """
    Prints a clean, human-readable evaluation dashboard.
    Run this after evaluate_research_output() to see the full picture.
    """
    C = eval_result["consistency_score"]
    Cv = eval_result["coverage"]
    F = eval_result["faithfulness"]
    D = eval_result["reasoning_depth"]
    O = eval_result["overall_score"]

    def bar(score, width=20):
        filled = round(score * width)
        return "█" * filled + "░" * (width - filled)

    print("\n" + "╔" + "═"*56 + "╗")
    print("║  DEEP RESEARCH EVALUATION DASHBOARD" + " "*19 + "║")
    print("╠" + "═"*56 + "╣")

    print(f"║  Consistency     {bar(C)}  {C:.0%}  ║")
    print(f"║  Coverage        {bar(Cv['score'])}  {Cv['score']:.0%}  ║")
    print(f"║  Faithfulness    {bar(F['score'])}  {F['score']:.0%}  ║")
    print(f"║  Reasoning depth {bar(D['score'])}  {D['score']:.0%}  ║")
    print("╠" + "═"*56 + "╣")
    print(f"║  OVERALL QUALITY {bar(O)}  {O:.0%}  ║")
    print("╚" + "═"*56 + "╝")

    print("\n── Coverage detail ──")
    for pq in Cv["per_question"]:
        status = "COVERED" if pq["covered"] else "MISSING"
        print(f"  [{status}] {pq['question']}")
        if not pq["covered"]:
            print(f"    Key terms not found: {set(pq['key_terms']) - set(pq['found_terms'])}")

    print("\n── Faithfulness detail ──")
    print(f"  {F['supported_sentences']}/{F['total_sentences']} sentences grounded in findings")
    if F["unsupported"]:
        print("  Potentially unsupported sentences:")
        for s in F["unsupported"][:3]:
            print(f"    - {s['sentence']}")

    print("\n── Reasoning depth detail ──")
    print(f"  Think block length: {D['word_count']} words")
    print(f"  Self-corrections  : {D['corrections_found'] or ['none detected']}")
    print(f"  Cross-references  : {len(D['crossrefs_found'])} found")



print("Now run:")
print("  eval_result = evaluate_research_output(research_result)")
print("  print_eval_dashboard(eval_result)")
print("  print(eval_result['best_report'])")

Now run:
  eval_result = evaluate_research_output(research_result)
  print_eval_dashboard(eval_result)
  print(eval_result['best_report'])


In [ ]:
from google.colab import drive
import json, os, datetime

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/DeepResearch_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

eval_result = evaluate_research_output(research_result)

report_path = f"{PROJECT_DIR}/report_{timestamp}.txt"
with open(report_path, 'w') as f:
    f.write(f"DEEP RESEARCH REPORT\n")
    f.write(f"Question: {research_result['question']}\n")
    f.write(f"Generated: {timestamp}\n")
    f.write("="*60 + "\n\n")
    f.write(eval_result['best_report'])
print(f"Report saved: {report_path}")

research_path = f"{PROJECT_DIR}/research_result_{timestamp}.json"
save_data = {
    "question":       research_result["question"],
    "sub_questions":  research_result["sub_questions"],
    "total_searches": research_result["total_searches"],
    "findings": {
        q: {"answer": d["answer"], "searches": d["searches"]}
        for q, d in research_result["findings"].items()
    }
}
with open(research_path, 'w') as f:
    json.dump(save_data, f, indent=2)
print(f"Research data saved: {research_path}")

eval_path = f"{PROJECT_DIR}/eval_scores_{timestamp}.json"
scores = {
    "timestamp":         timestamp,
    "question":          research_result["question"],
    "overall_score":     eval_result["overall_score"],
    "consistency":       eval_result["consistency_score"],
    "coverage":          eval_result["coverage"]["score"],
    "faithfulness":      eval_result["faithfulness"]["score"],
    "reasoning_depth":   eval_result["reasoning_depth"]["score"],
    "sub_questions":     research_result["sub_questions"],
    "total_searches":    research_result["total_searches"],
    "covered_count":     eval_result["coverage"]["covered"],
    "total_sentences":   eval_result["faithfulness"]["total_sentences"],
    "supported_sentences": eval_result["faithfulness"]["supported_sentences"],
}
with open(eval_path, 'w') as f:
    json.dump(scores, f, indent=2)
print(f"Eval scores saved: {eval_path}")

print("\nAll outputs saved to Google Drive successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

EVALUATION STARTING

[1/4] Running self-consistency (3 synthesis runs)...

  Run 1/3... 

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x78bf58df17c0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:
PROJECT_INFO = """
╔══════════════════════════════════════════════════════╗
║           DEEP RESEARCH CAPABILITY                   ║
║           Project 4 — Reasoning Models              ║
╠══════════════════════════════════════════════════════╣
║  Model:     DeepSeek-R1 7B (via Ollama on Colab)    ║
║  Platform:  Google Colab Free Tier (T4 GPU)          ║
╠══════════════════════════════════════════════════════╣
║  NOTEBOOK STRUCTURE:                                 ║
║  Section 1 (Cells 1-7)   — Setup & first query      ║
║  Section 2 (Cells 8-13)  — ReAct + Tavily search    ║
║  Section 3 (Cells 14-19) — Deep research pipeline   ║
║  Section 4 (Cells 20-28) — Evaluation framework     ║
║  Section 5 (Cells 29-30) — Save outputs & wrap-up   ║
╠══════════════════════════════════════════════════════╣
║  KEY CONCEPTS DEMONSTRATED:                          ║
║  • Reasoning models (DeepSeek-R1, <think> blocks)   ║
║  • Inference-time scaling (self-consistency, N=3)   ║
║  • Chain-of-Thought prompting (ReAct pattern)        ║
║  • Deep research pipeline (plan→search→synthesise)  ║
║  • Evaluation metrics (coverage, faithfulness,       ║
║    reasoning depth, consistency)                     ║
╚══════════════════════════════════════════════════════╝
"""
print(PROJECT_INFO)

In [ ]:
README = """
# Deep Research Capability — Project 4

## What this project does
An end-to-end deep research agent built on DeepSeek-R1 (local, free)
that takes a complex research question and produces an evaluated,
multi-source research report.

## Architecture
User question
    │
    ▼
[Planner]  — DeepSeek-R1 decomposes into 3-5 sub-questions (JSON output)
    │
    ▼
[Researcher × N]  — ReAct loop per sub-question:
                    R1 thinks → Tavily search → R1 reasons → repeat
    │
    ▼
[Synthesiser]  — R1 reads all findings, writes structured report
    │
    ▼
[Evaluator]  — 4 metrics: coverage, faithfulness, depth, consistency

## Key concepts demonstrated
- Reasoning model: DeepSeek-R1 produces visible <think> blocks before answering
- Inference-time scaling: self-consistency (Best-of-3) at synthesis stage
- Chain-of-Thought: ReAct pattern (Thought → Action → Observation loop)
- Evaluation: 4 automated metrics scored 0-1, combined into overall quality score

## Setup
1. Google Colab free tier (T4 GPU)
2. Run: !curl -fsSL https://ollama.com/install.sh | sh
3. Run: !ollama pull deepseek-r1:7b
4. Get free Tavily API key at app.tavily.com
5. Run notebook cells in order (Sections 1-5)

## Results
- Typical report: 400-600 words, 3-5 sections
- Typical eval score: 70-85% overall quality
- Typical run time: 20-30 minutes end-to-end on Colab free tier
- Tavily API usage: 8-15 searches per report (well within 1000/month free tier)

## Files
- deep_research_notebook.ipynb  — main notebook
- report_[timestamp].txt        — generated research reports
- research_result_[timestamp].json — full pipeline output
- eval_scores_[timestamp].json  — evaluation scores
"""

print(README)

readme_path = f"{PROJECT_DIR}/README.md"
with open(readme_path, 'w') as f:
    f.write(README)
print(f"README saved to {readme_path}")